In [5]:
import os
import numpy as np
import torch

from cosmos_predict2.utils.printer import print_batch
from cosmos_predict2.utils.vis_helpers import save_action_as_image
from imaginaire.utils.io import save_image_or_video
from imaginaire.lazy_config import instantiate

from cosmos_predict2.configs.expert.defaults.data_tcl import tcl_train_dataset, DataLoader

In [6]:
train_dataset = instantiate(
    tcl_train_dataset,
    p_camera_drop=0.,
    camera_keys=["image", "gripper"]
)
train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=16,
    drop_last=True
)

[TCLDataset] loaded key=rel_actions shape=(28368, 7) from /home/geyuan/local_soft/TCL/1009_spoon_pick_place/extracted/rel_actions.npy
[TCLDataset] total length: 28368
[TCLDatasetHDF5] using h5 data: /home/geyuan/local_soft/TCL/hdf5/1009_spoon_pick_place_240p.h5 . total_len: 28368
[TCLDataset] loading dataset statistics from: /home/geyuan/local_soft/TCL/1009_spoon_pick_place/statistics.json
[DEBUG] Loading language embeddings from: /home/geyuan/local_soft/TCL/lang_emb_t5xxl/all/t5_embeddings.npz
[TCLImageDataset] dataset loaded, split=train, val_ratio=0.0, len=28368; meta_total_len=28368, norm_type=minmax, action_min=[-0.09842834 -0.09215698 -0.09136963 -0.78527469 -0.73214177 -0.69676394
  0.        ], action_max=[0.09372559 0.09450684 0.09607849 1.02278088 0.8995309  2.03223691
 1.        ], action_mean=[-2.65218172e-04  8.43159331e-04 -5.51890090e-04  4.54934811e-03
  7.11483133e-04  3.08984176e-03  3.40383531e-01], action_std=[0.03206969 0.02588468 0.02548615 0.05496435 0.0512122  0

In [11]:
from tqdm import tqdm

vis_idx = 1
in_sample = None

for idx, batch in enumerate(tqdm(train_dataloader)):
    if idx == 0:
        print_batch('TCL', batch)
    if idx < vis_idx:
        continue
    print_batch('TCL', batch)
    # print("language:", batch["task"]["language_instruction"])

    in_sample = batch
    break

  0%|                                                                                      | 1/3546 [00:01<1:14:14,  1.26s/it]

TCL: Dict, keys=['action', 'video', 'agent_pos', 'annotation_file', '__key__', 'lang_text', 't5_text_embeddings', 't5_text_mask', 'fps', 'image_size', 'num_frames', 'padding_mask', 'sample_n_views', 'view_indices', 'latent_view_indices_B_T']
--action, <class 'torch.Tensor'>, shape=torch.Size([8, 20, 7]), min=-1.0000, max=1.0000
--video, <class 'torch.Tensor'>, shape=torch.Size([8, 3, 50, 128, 160]), min=0.0000, max=255.0000
--agent_pos, <class 'torch.Tensor'>, shape=torch.Size([8, 25, 8]), min=-3.0338, max=0.8804
--annotation_file: List, len=8, elem_type=<class 'str'>
----[0]: <class 'str'>, len=4, value='None'
--__key__: List, len=8, elem_type=<class 'str'>
----[0]: <class 'str'>, len=4, value='None'
--lang_text: List, len=8, elem_type=<class 'str'>
----[0]: <class 'str'>, len=68, value='Pick up the spoon in the cup, and then place the spoon into a bowl.#'
--t5_text_embeddings, <class 'torch.Tensor'>, shape=torch.Size([8, 512, 1024]), min=-0.5859, max=0.6602
--t5_text_mask, <class 'to

  0%|                                                                                      | 1/3546 [00:02<2:25:02,  2.45s/it]


In [12]:
""" Visualization Remapped Dataloader """
max_vis_len = 50
mv_sample = in_sample
DEBUG_DATASET = "tcl"

horizon = mv_sample['agent_pos'][0].shape[0]
save_image_or_video(
    mv_sample['video'][0, :, :horizon].float() / 255.,
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_agentview.mp4",
    fps=15
)
save_image_or_video(
    mv_sample['video'][0, :, horizon:].float() / 255.,
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_gripper.mp4",
    fps=15
)

action_meta = train_dataset.dataset_meta_dict
action_min = action_meta['min']
action_max = action_meta['max']

def denorm_action(act):
    # return (act + 1) / 2 * (np.array(meta_p99) - np.array(meta_p01)) + np.array(meta_p01)
    return (act + 1) / 2 * (action_max - action_min) + action_min

save_action_as_image(
    denorm_action(mv_sample['action'])[0, :, :3],
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_action012.png",
)
save_action_as_image(
    denorm_action(mv_sample['action'])[0, :, 6:7],
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_action6.png",
)

save_action_as_image(
    mv_sample['agent_pos'][0, :, :3],
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_agentpos012.png",
)
# save_action_as_image(
#     mv_sample['agent_pos'][:, -1:] * dataset_multi_view.meta_gripper_states_std[-1] + dataset_multi_view.meta_gripper_states_mean[-1],
#     "/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_agentpos6.png",
# )


[DEBUG] save_3d_action_as_image: torch.Size([20, 3]) torch.float64 tensor(0., dtype=torch.float64) tensor(0., dtype=torch.float64)
[10-15 16:58:34|INFO|../../../../../../../home/geyuan/code/cospred2nvidia/cosmos_predict2/utils/vis_helpers.py:185:save_3d_action_as_image] Saved 3D action trajectory visualization to /home/geyuan/code/cospred2nvidia/output/de_tcl_mv_action012.png
[DEBUG] save_1d_action_as_image: torch.Size([20, 1]) torch.float64 tensor(0., dtype=torch.float64) tensor(0., dtype=torch.float64)
Saved 1D action trajectory visualization to /home/geyuan/code/cospred2nvidia/output/de_tcl_mv_action6.png
[DEBUG] save_3d_action_as_image: torch.Size([25, 3]) torch.float32 tensor(-0.1693) tensor(0.5024)
[10-15 16:58:35|INFO|../../../../../../../home/geyuan/code/cospred2nvidia/cosmos_predict2/utils/vis_helpers.py:185:save_3d_action_as_image] Saved 3D action trajectory visualization to /home/geyuan/code/cospred2nvidia/output/de_tcl_mv_agentpos012.png
